# Derivation: Dimensionless Oxygen Diffusion and Reaction

How does an oxygen balance in a tissue become a system of linear algebraic equations? In this companion to [the L6d lab](CHEME-5800-L6d-Lab-IterativeLinearSolvers-Fall-2026.ipynb), we will derive the diffusion–reaction equation, identify its dimensionless parameters, and approximate its spatial derivatives on a grid. Each matrix row will represent the steady oxygen balance at one interior point.

> __Learning Objectives:__
>
> By the end of this derivation, you should be able to:
>
> * __Derive the oxygen balance:__ Diffusive transport supplies oxygen while cells consume it throughout the tissue. We'll start with a material balance and use Fick's law and first-order uptake to obtain the governing partial differential equation.
> * __Identify the dimensionless competition:__ The Thiele modulus combines the tissue length, diffusivity, and consumption coefficient. We'll nondimensionalize the balance and interpret its square as a ratio of characteristic times.
> * __Construct the linear algebraic equations:__ A finite-difference approximation relates each interior concentration to its neighbors. We'll move known boundary values to the right-hand side, assemble a small matrix explicitly, and connect the result to the L6c solvers.

Let's get started!

___


## Geometry, assumptions, and the oxygen balance

Consider a square tissue slice with coordinates $-L\leq x,y\leq L$, where $L>0$ is the half-width in $\mathrm{m}$. Its broad faces are impermeable, and we assume the oxygen concentration is uniform through its thickness. Oxygen enters through the four lateral edges, each maintained at a prescribed concentration $c_s\geq0$ in $\mathrm{mol\,m^{-3}}$. This geometry gives a two-dimensional diffusion problem.

Let $c(x,y,t)$ be the oxygen concentration, where $t$ is time in $\mathrm{s}$. We treat the tissue as a stationary, homogeneous medium with constant effective diffusivity $D>0$ in $\mathrm{m^2\,s^{-1}}$. Cells consume oxygen at a volumetric rate $R(c)=k_rc$, with constant $k_r\geq0$ in $\mathrm{s^{-1}}$. Cell density and per-cell uptake kinetics are uniform throughout the patch and remain fixed; both are included in the effective coefficient $k_r$.


<style>
.course-diagram { color-scheme: light dark; }
:host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
:host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
body[data-vscode-theme-kind="vscode-light"] .course-diagram,
body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram,
body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
:host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
:host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram,
body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
@media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
<img class="course-diagram" src="figs/Fig-Tissue-Stencil.svg" width="960" style="max-width:100%; height:auto;" alt="A square tissue slice with uniform cell density receives oxygen at its four edges and consumes it throughout its interior. A five-point stencil couples each interior concentration to its four nearest neighbors.">
</p>

__Diffusion supplies oxygen to a consuming interior.__ The edge concentration is imposed by the surrounding reservoir. The model assumes negligible external resistance between that reservoir and the tissue edge.

Let $J_x$ and $J_y$ denote the diffusive molar fluxes in the positive $x$ and $y$ directions, in $\mathrm{mol\,m^{-2}\,s^{-1}}$. For a small region of width $\Delta x$, height $\Delta y$, and thickness $H$, accumulation equals transport in minus transport out minus consumption. Dividing by $H\Delta x\Delta y$ gives the local balance in the limit:

$$
\begin{aligned}
\frac{\partial c}{\partial t}
&=\lim_{\Delta x\to0}\frac{J_x(x,y,t)-J_x(x+\Delta x,y,t)}{\Delta x}
 +\lim_{\Delta y\to0}\frac{J_y(x,y,t)-J_y(x,y+\Delta y,t)}{\Delta y}
 -k_rc,\\
&=-\frac{\partial J_x}{\partial x}-\frac{\partial J_y}{\partial y}-k_rc.
\end{aligned}
$$

Fick's law relates the flux to the concentration gradient:

$$
J_x=-D\frac{\partial c}{\partial x},\qquad
J_y=-D\frac{\partial c}{\partial y}.
$$

Substituting these fluxes into the balance, with constant $D$, gives:

$$
\boxed{
\frac{\partial c}{\partial t}
=D\left(\frac{\partial^2c}{\partial x^2}+\frac{\partial^2c}{\partial y^2}\right)-k_rc.
}
$$

Every term has units $\mathrm{mol\,m^{-3}\,s^{-1}}$. The negative reaction term removes oxygen. The boundary condition is $c=c_s$ at $x=\pm L$ and $y=\pm L$. A transient calculation would also require an initial concentration field; the lab seeks the steady field.

### Why use first-order consumption?

First-order uptake makes the reaction term linear in the unknown concentration. It is also the low-concentration approximation to a saturating uptake law. If $R_{\max}$ is the maximum volumetric consumption rate and $K_M>0$ is the half-saturation concentration, then:

$$
R(c)=R_{\max}\frac{c}{K_M+c}
\approx\frac{R_{\max}}{K_M}c=k_rc,
\qquad c\ll K_M.
$$

The units of $R_{\max}/K_M$ are $\mathrm{s^{-1}}$. [Ehsan and George (2013)](https://pmc.ncbi.nlm.nih.gov/articles/PMC3638538/) study oxygen diffusion with saturating cellular uptake in engineered tissue. Here we adopt constant first-order uptake as a teaching model; applying it quantitatively requires checking the concentration range and estimating its coefficient.

___


## Nondimensionalization and the Thiele modulus

Choose a fixed reference concentration $c_{\mathrm{ref}}>0$ and scale distance with the half-width $L$. Use the diffusion time $L^2/D$ to scale physical time. Define the dimensionless variables by:

$$
\xi=\frac{x}{L},\qquad
\eta=\frac{y}{L},\qquad
\tau=\frac{Dt}{L^2},\qquad
\theta(\xi,\eta,\tau)=\frac{c(x,y,t)}{c_{\mathrm{ref}}}.
$$

Thus, the square becomes $-1\leq\xi,\eta\leq1$. The chain rule transforms the derivatives as follows:

$$
\begin{aligned}
\frac{\partial c}{\partial t}
 &=\frac{Dc_{\mathrm{ref}}}{L^2}\frac{\partial\theta}{\partial\tau},\\
\frac{\partial^2c}{\partial x^2}
 &=\frac{c_{\mathrm{ref}}}{L^2}\frac{\partial^2\theta}{\partial\xi^2},\qquad
\frac{\partial^2c}{\partial y^2}
 =\frac{c_{\mathrm{ref}}}{L^2}\frac{\partial^2\theta}{\partial\eta^2}.
\end{aligned}
$$

Substitute these expressions into the dimensional balance and divide by $Dc_{\mathrm{ref}}/L^2$ to obtain:

$$
\begin{aligned}
\frac{Dc_{\mathrm{ref}}}{L^2}\frac{\partial\theta}{\partial\tau}
&=\frac{Dc_{\mathrm{ref}}}{L^2}
\left(\frac{\partial^2\theta}{\partial\xi^2}+\frac{\partial^2\theta}{\partial\eta^2}\right)
-k_rc_{\mathrm{ref}}\theta,\\
\frac{\partial\theta}{\partial\tau}
&=\frac{\partial^2\theta}{\partial\xi^2}+\frac{\partial^2\theta}{\partial\eta^2}
-\phi^2\theta,\qquad \phi^2=\frac{k_rL^2}{D}.
\end{aligned}
$$

> __What does the Thiele modulus measure?__
>
> With the characteristic length defined as the square's half-width, define the diffusion time $t_D$ and reaction time $t_R$. For $k_r>0$, they give the first-order Thiele modulus as follows:
>
> $$
> \begin{aligned}
> t_D&=\frac{L^2}{D}, &&\text{diffusion time},\\
> t_R&=\frac{1}{k_r}, &&\text{reaction time},\\
> \phi&=L\sqrt{\frac{k_r}{D}},\qquad
> \phi^2=\frac{k_rL^2}{D}=\frac{t_D}{t_R}.
> \end{aligned}
> $$
>
> When $\phi\ll1$, diffusion redistributes oxygen rapidly compared with its consumption. When $\phi\gg1$, consumption is rapid relative to diffusion over the half-width, and larger concentration gradients develop. If $k_r=0$, then $\phi=0$ and oxygen is not consumed.

The same diffusion–reaction scaling appears in [Wittrup's MIT reaction-engineering notes](https://ocw.mit.edu/courses/10-37-chemical-and-biological-reaction-engineering-spring-2007/resources/lec20_04272007_w/). A reported Thiele modulus must specify its characteristic length: using the full width $2L$ would double its numerical value.

The dimensionless boundary concentration is $\beta=c_s/c_{\mathrm{ref}}$, so $\theta=\beta$ on all four edges. In the baseline lab, $c_s=c_{\mathrm{ref}}$ and $\beta=1$. Keeping $c_{\mathrm{ref}}$ fixed across scenarios means that halving the edge concentration changes $\beta$ to $1/2$.

___


## The steady dimensionless boundary-value problem

At steady state, accumulation vanishes, so $\partial\theta/\partial\tau=0$. Let $\Omega=(-1,1)^2$ denote the interior of the dimensionless square, $\partial\Omega$ its boundary, and $\nabla_*^2=\partial^2/\partial\xi^2+\partial^2/\partial\eta^2$ its Laplacian. Multiplying the steady balance by $-1$ gives:

> __Steady oxygen diffusion and consumption:__
>
> Given $\phi\geq0$ and edge concentration $\beta\geq0$, find the dimensionless concentration field $\theta$ satisfying:
>
> $$
> \boxed{
> \begin{aligned}
> -\nabla_*^2\theta+\phi^2\theta&=0,
> &&(\xi,\eta)\in\Omega,\\
> \theta&=\beta,
> &&(\xi,\eta)\in\partial\Omega.
> \end{aligned}
> }
> $$
>
> Diffusive transport balances consumption at every interior point. Both the differential equation and its prescribed concentration boundary conditions are linear in $\theta$.

For $\phi=0$, the constant field $\theta=\beta$ satisfies the problem. For positive $\phi$ and $\beta$, consumption lowers the interior concentration below the edge concentration. Linearity also gives a useful check: at fixed $\phi$, halving $\beta$ halves the complete steady field.

___


## From spatial derivatives to grid equations

Place $n\geq1$ interior points along each coordinate direction and include the two edge points. Define the dimensionless grid spacing and coordinates by:

$$
h=\frac{2}{n+1},\qquad
\xi_i=-1+ih,\qquad \eta_j=-1+jh,
\qquad i,j=0,1,\ldots,n+1.
$$

Let $\theta_{i,j}$ approximate the steady concentration at $(\xi_i,\eta_j)$. Interior indices satisfy $1\leq i,j\leq n$; an index of zero or $n+1$ places a point on the boundary.

### Centered differences

At a smooth interior point, Taylor expansions in the horizontal direction give:

$$
\begin{aligned}
\theta(\xi_i+h,\eta_j)
&=\theta(\xi_i,\eta_j)+h\theta_{\xi}
+\frac{h^2}{2}\theta_{\xi\xi}+\frac{h^3}{6}\theta_{\xi\xi\xi}+O(h^4),\\
\theta(\xi_i-h,\eta_j)
&=\theta(\xi_i,\eta_j)-h\theta_{\xi}
+\frac{h^2}{2}\theta_{\xi\xi}-\frac{h^3}{6}\theta_{\xi\xi\xi}+O(h^4),
\end{aligned}
$$

where the derivatives on the right are evaluated at $(\xi_i,\eta_j)$. Adding the expansions cancels the odd-order terms. Rearranging and dividing by $h^2$ gives the centered second derivative; applying the same calculation vertically gives:

$$
\begin{aligned}
\left.\frac{\partial^2\theta}{\partial\xi^2}\right|_{i,j}
&=\frac{\theta(\xi_{i-1},\eta_j)-2\theta(\xi_i,\eta_j)+\theta(\xi_{i+1},\eta_j)}{h^2}+O(h^2),\\
\left.\frac{\partial^2\theta}{\partial\eta^2}\right|_{i,j}
&=\frac{\theta(\xi_i,\eta_{j-1})-2\theta(\xi_i,\eta_j)+\theta(\xi_i,\eta_{j+1})}{h^2}+O(h^2).
\end{aligned}
$$

Substituting the difference approximations into the steady equation defines the discrete balance:

$$
-\frac{\theta_{i-1,j}-2\theta_{i,j}+\theta_{i+1,j}}{h^2}
-\frac{\theta_{i,j-1}-2\theta_{i,j}+\theta_{i,j+1}}{h^2}
+\phi^2\theta_{i,j}=0.
$$

Multiply each equation by $h^2$ and collect the central concentration to obtain the five-point stencil:

$$
\boxed{
(4+\phi^2h^2)\theta_{i,j}
-\theta_{i-1,j}-\theta_{i+1,j}-\theta_{i,j-1}-\theta_{i,j+1}=0.
}
$$

The four diffusion contributions give the diagonal term 4. Consumption adds $\phi^2h^2$. Each neighbor has coefficient $-1$, so increasing a neighboring concentration increases the diffusive supply to the central point.

### Incorporating the boundary values

At the lower-left interior point $(1,1)$, the neighbors $(0,1)$ and $(1,0)$ lie on supplied edges and have concentration $\beta$. For $n\geq2$, its equation is therefore:

$$
\begin{aligned}
(4+\phi^2h^2)\theta_{1,1}-\beta-\theta_{2,1}-\beta-\theta_{1,2}&=0,\\
(4+\phi^2h^2)\theta_{1,1}-\theta_{2,1}-\theta_{1,2}&=2\beta.
\end{aligned}
$$

Each known boundary neighbor contributes $+\beta$ to the right-hand side. A point beside one edge receives $\beta$; a point beside two edges receives $2\beta$; a point with four interior neighbors receives zero. For the special grid $n=1$, the only interior point has four boundary neighbors and receives $4\beta$.

___


## Assembling the linear algebraic system

Number the interior concentrations with the horizontal index changing first. Define the vector position $p$ and the corresponding unknown by:

$$
p=i+(j-1)n,\qquad
\boldsymbol{\theta}_p=\theta_{i,j},
\qquad 1\leq i,j\leq n.
$$

The unknown vector $\boldsymbol{\theta}\in\mathbb{R}^{n^2}$ contains only interior values. Its system matrix $\mathbf{A}\in\mathbb{R}^{n^2\times n^2}$ and right-hand side $\mathbf{b}\in\mathbb{R}^{n^2}$ are assembled as follows:

| Contribution to row $p$ | Matrix or right-hand-side entry |
|---|---|
| Central concentration | $A_{p,p}=4+\phi^2h^2$ |
| Left interior neighbor, if $i>1$ | $A_{p,p-1}=-1$ |
| Right interior neighbor, if $i<n$ | $A_{p,p+1}=-1$ |
| Bottom interior neighbor, if $j>1$ | $A_{p,p-n}=-1$ |
| Top interior neighbor, if $j<n$ | $A_{p,p+n}=-1$ |
| Each prescribed boundary neighbor | Add $\beta$ to $b_p$ |

All other matrix entries are zero. The conditions on $i$ prevent the last point of one horizontal row from being incorrectly connected to the first point of the next row. Collecting every grid balance gives the dimensionless linear algebraic equations:

$$
\boxed{\mathbf{A}(\phi,h)\boldsymbol{\theta}=\mathbf{b}(\beta).}
$$

### A four-unknown example

For $n=2$, the spacing is $h=2/3$, and the ordering is $(1,1),(2,1),(1,2),(2,2)$. Write $a=4+\phi^2h^2$ for the diagonal coefficient. Every interior point has two boundary neighbors, so the complete system is:

$$
\begin{bmatrix}
a&-1&-1&0\\
-1&a&0&-1\\
-1&0&a&-1\\
0&-1&-1&a
\end{bmatrix}
\begin{bmatrix}
\theta_{1,1}\\\theta_{2,1}\\\theta_{1,2}\\\theta_{2,2}
\end{bmatrix}
=
\begin{bmatrix}
2\beta\\2\beta\\2\beta\\2\beta
\end{bmatrix}.
$$

Symmetry makes the four concentrations equal. Substituting a common value $q$ gives $(a-2)q=2\beta$, hence $q=2\beta/(2+\phi^2h^2)$. This small example provides an independent check of the matrix assembly in [src/Compute.jl](src/Compute.jl).

___


## Connecting the system to the iterative methods

For $\phi>0$, the diagonal $4+\phi^2h^2$ exceeds the sum of the magnitudes of the off-diagonal entries, which is at most four. The matrix is symmetric, strictly diagonally dominant, and has a positive diagonal. It is therefore positive definite. The L6c results guarantee convergence of Jacobi and Gauss–Seidel, and convergence of successive over-relaxation for $0<\omega<2$.

For completeness, the no-consumption case $\phi=0$ also converges on this grid with prescribed boundary values. The eigenvalues of its discrete diffusion–reaction matrix are:

$$
\lambda_{r,s}(\mathbf{A})
=4+\phi^2h^2-2\cos\left(\frac{r\pi}{n+1}\right)
-2\cos\left(\frac{s\pi}{n+1}\right)>0,
\qquad r,s=1,\ldots,n,
$$

where $r$ and $s$ index the grid modes. The Jacobi iteration matrix has spectral radius:

$$
\rho(\mathbf{G}_J)
=\frac{4\cos\left(\pi/(n+1)\right)}{4+\phi^2h^2}<1.
$$

Thus Jacobi converges even at $\phi=0$, where interior rows can be only weakly diagonally dominant. Positive definiteness gives the Gauss–Seidel and SOR guarantees as well.

Let $m$ count numerical corrections and let $\mathcal{I}(i,j)$ denote the interior neighbors of point $(i,j)$. Rearranging row $p=i+(j-1)n$ gives the Jacobi update:

$$
\theta_{i,j}^{(m+1)}
=\frac{b_p+\displaystyle\sum_{(u,v)\in\mathcal{I}(i,j)}\theta_{u,v}^{(m)}}
{4+\phi^2h^2}.
$$

Gauss–Seidel uses the newly updated concentrations as soon as they are available. Successive over-relaxation blends the previous concentration with its Gauss–Seidel candidate using $\omega$. These are numerical corrections to the steady solution. Predicting physical time evolution would require discretizing the accumulation term as well.

The formula for $\rho(\mathbf{G}_J)$ also explains two trends. At fixed grid spacing, increasing $\phi$ strengthens the diagonal and reduces the Jacobi spectral radius. At fixed $\phi$, refining the grid makes that radius approach one, so Jacobi generally needs more corrections.

___


## Checking the result and the grid approximation

We can check a computed concentration vector $\widehat{\boldsymbol{\theta}}$ by evaluating its residual and comparing it with a direct solve of the same grid equations:

$$
\mathbf{r}=\mathbf{b}-\mathbf{A}\widehat{\boldsymbol{\theta}},\qquad
\Delta\boldsymbol{\theta}=\widehat{\boldsymbol{\theta}}-\boldsymbol{\theta}_{\mathrm{direct}}.
$$

The residual measures agreement with the algebraic balances. The difference from the direct solution checks the iterative solver independently. Both refer to the same spatial approximation.

The centered derivative approximation has local truncation error of order $h^2$ where the solution is sufficiently smooth. To examine spatial accuracy, refine the grid while keeping $\phi$ and $\beta$ fixed and compare a physical observable, such as the center concentration. Replacing $n$ by $2n+1$ halves $h$ and preserves the center point when $n$ is odd. A small algebraic residual alone does not measure this grid error.

We multiplied every equation by $h^2$. If $\mathbf{r}_{\mathrm{PDE}}$ denotes the residual of the unscaled difference equations, the two residual vectors satisfy:

$$
\mathbf{r}=h^2\mathbf{r}_{\mathrm{PDE}}.
$$

Use the same matrix scaling and grid when comparing the three solver histories. When refining the grid, account for the changed scaling and number of equations if comparing residual norms.

Finally, check the physical structure of the solution. With no consumption, the concentration should equal the supplied edge value everywhere. With positive consumption and edge supply, the interior concentration should remain positive and below the edge value. Halving $\beta$ at fixed $\phi$ should halve the complete field. These checks test consequences of the model in addition to the solver's stopping criterion.

___


## Summary

We derived the steady dimensionless linear system used in the oxygen-delivery lab.

> __Key Takeaways:__
>
> * __A material balance gives the diffusion–reaction equation:__ We combined accumulation, diffusive transport, and first-order oxygen consumption. The geometry and prescribed edge concentrations completed the model.
> * __The Thiele modulus compares diffusion and reaction:__ We scaled the tissue with its half-width and a fixed concentration reference. The resulting parameters separate the competition between diffusion and consumption from the oxygen supplied at the edges.
> * __Grid balances become matrix rows:__ We replaced spatial derivatives with centered differences, numbered the interior concentrations, and moved known edge values to the right-hand side. This produced the linear system solved by the L6c methods, with separate checks for algebraic convergence and grid accuracy.

Return to [the L6d lab](CHEME-5800-L6d-Lab-IterativeLinearSolvers-Fall-2026.ipynb) to compute the fields and change the tissue parameters.

___
